# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [10]:
links = fetch_website_links("https://raisin.com/en/")
links

['/en/corporate/',
 '/en/corporate/press/',
 '/en/corporate/careers/',
 '/en/login/',
 '/en/corporate/about-raisin/',
 'https://www.raisin.bank/en/',
 '/en/corporate/become-a-partner-bank/',
 '/en/corporate/become-a-distribution-partner/',
 '/en/corporate/press/',
 '/en/corporate/careers/',
 '/en/login/',
 'https://www.raisin.com/en/select-country/',
 'https://www.raisin.com/en/corporate/become-a-partner-bank/',
 'https://www.raisin.com/en/corporate/become-a-distribution-partner/',
 'https://www.raisin.com/en/corporate/about-raisin/',
 'https://www.linkedin.com/company/join-raisin/',
 '/en/corporate/about-raisin/',
 'https://www.raisin.bank/en/',
 '/en/corporate/careers',
 '/en/corporate/press',
 '/en/corporate/become-a-partner-bank',
 '/en/corporate/become-a-distribution-partner',
 '/en/corporate/become-an-affiliate-partner',
 '/en/corporate/contact',
 'https://help.raisin.com/hc/en-us/',
 '/en/corporate/about-raisin/',
 'https://www.raisin.bank/en/',
 '/en/corporate/careers',
 '/en/c

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://raisin.com/en/"))


Here is the list of links on the website https://raisin.com/en/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/en/corporate/
/en/corporate/press/
/en/corporate/careers/
/en/login/
/en/corporate/about-raisin/
https://www.raisin.bank/en/
/en/corporate/become-a-partner-bank/
/en/corporate/become-a-distribution-partner/
/en/corporate/press/
/en/corporate/careers/
/en/login/
https://www.raisin.com/en/select-country/
https://www.raisin.com/en/corporate/become-a-partner-bank/
https://www.raisin.com/en/corporate/become-a-distribution-partner/
https://www.raisin.com/en/corporate/about-raisin/
https://www.linkedin.com/company/join-raisin/
/en/corporate/about-raisin/
https://www.raisin.bank/en/
/en/corporate/careers
/en/corporate/press
/en/corporate/become-a-partner-bank
/en/corporate/become-a-distribution-partne

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print(f"{result}")
    links = json.loads(result)
    return links
    

In [14]:
select_relevant_links("https://raisin.com/en/")

{'links': [{'type': 'company page',
   'url': 'https://www.raisin.com/en/corporate/'},
  {'type': 'about page',
   'url': 'https://www.raisin.com/en/corporate/about-raisin/'},
  {'type': 'press page', 'url': 'https://www.raisin.com/en/corporate/press/'},
  {'type': 'careers page',
   'url': 'https://www.raisin.com/en/corporate/careers/'},
  {'type': 'partner bank page',
   'url': 'https://www.raisin.com/en/corporate/become-a-partner-bank/'},
  {'type': 'distribution partner page',
   'url': 'https://www.raisin.com/en/corporate/become-a-distribution-partner/'},
  {'type': 'affiliate partner page',
   'url': 'https://www.raisin.com/en/corporate/become-an-affiliate-partner/'},
  {'type': 'contact page',
   'url': 'https://www.raisin.com/en/corporate/contact'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/join-raisin/'}]}

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [16]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn hub', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
about 15 hours ago
•
115k
•
474
deepseek-ai/DeepSeek-V4-Pro
Updated
3 days ago
•
1.17M
•
3.77k
Zyphra/ZAYA1-8B
Updated
about 16 hours ago
•
23.6k
•
308
SeeSee21/Z-Anime
Updated
13 days ago
•
8.43k
•
249
openai/privacy-filter
Updated
17 days ago
•
180k
•
1.38k
Browse 2M+ models
Spaces
Running
on
Zero
MCP
345
Qwen Image Edit + Loras built-in
👀
345
Qwen image edit with 🔞 loras
Running
on
Zero
MCP
1.04k
Wan2.2 14B Fast Preview
🐌
1.04k
generate a video from an image with a text prompt
Running
98
The ulti

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\nabout 15 hours ago\n•\n115k\n•\n474\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n3 days ago\n•\n1.17M\n•\n3.77k\nZyphra/ZAYA1-8B\nUpdated\nabout 16 hours ago\n•\n23.6k\n•\n309\nSeeSee21/Z-Anime\nUpdated\n13 days ago\n•\n8.43k\n•\n249\nopenai/privacy-filter\nUpdated\n17 days ago\n•\n180k\n•\n1.38k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n345\nQwen I

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [23]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI community and collaboration platform dedicated to building the future of machine learning (ML). It serves as the central hub for ML enthusiasts, engineers, data scientists, and researchers to share, explore, discover, and experiment with open-source models, datasets, and applications.

At its core, Hugging Face empowers the next generation of ML professionals to learn, collaborate, and innovate together — driving an open and ethical AI future. The platform currently hosts over 2 million models and 500,000+ datasets spanning multiple modalities such as text, image, video, audio, and 3D.

---

## Our Platform and Offerings

- **Hugging Face Hub:**  
  A centralized space for unlimited public hosting and collaboration on ML assets including models, datasets, and applications.

- **Models:**  
  Explore and share millions of pre-trained models for diverse AI tasks. Popular trending models include SulphurAI’s Sulphur-2-base, deepseek-ai’s DeepSeek-V4-Pro, and others constantly updated by the community.

- **Datasets:**  
  Access hundreds of thousands of datasets curated for various machine learning needs, such as AgentTrove or Zero-To-CAD-1m.

- **Spaces:**  
  Interactive ML applications actively running on Hugging Face infrastructure, enabling users to test and deploy AI-powered tools like image editing or video generation from text prompts.

- **Open Source Stack:**  
  Tools allowing faster development, experimentation, and deployment of AI technology.

- **Enterprise Solutions:**  
  Tailored plans and services for businesses looking to leverage AI efficiently at scale.

---

## Company Culture

- **Community & Collaboration:**  
  Hugging Face fosters a welcoming, inclusive community where sharing knowledge and resources accelerates AI innovation.

- **Open & Ethical AI:**  
  The company champions transparency and ethics in AI, encouraging responsible use and development of machine learning technologies.

- **Cutting-Edge Research & Development:**  
  Hugging Face’s talented science team continuously explores frontier technologies, keeping the platform at the heart of the AI revolution.

- **Empowerment:**  
  Enables machine learning practitioners of all skill levels to build their portfolios, share their work publicly, and grow professionally.

---

## Customers and Community

Hugging Face serves millions of users worldwide, including:

- Individual ML engineers, data scientists, and researchers  
- Academic institutions fueling AI research  
- Tech startups and enterprises deploying AI at scale  
- Open-source contributors enriching the AI ecosystem

The platform’s tools and community accelerate AI development from prototype to production, making it a trusted partner in the ML space.

---

## Careers at Hugging Face

Join a passionate and innovative team shaping the future of AI! Hugging Face offers exciting career opportunities across multiple disciplines, including:

- Machine Learning Engineering  
- Research Science  
- Software Development  
- Community Management  
- Product Management  
- Enterprise Solutions

The company values diversity, encourages continuous learning, and supports a culture of creativity, ethics, and impact.

---

## Connect & Get Involved

- Explore AI models, datasets, and applications at [huggingface.co](https://huggingface.co)  
- Join the community on GitHub, Twitter, LinkedIn, and Discord  
- Contribute to open source projects or build your own portfolio  
- Learn from comprehensive documentation, forums, and blog content  
- Discover career opportunities and become part of the AI future

---

**Hugging Face** – *The AI Community Building the Future*  
Together, we create, collaborate, and innovate to advance the world of machine learning.

---

### Brand Colors  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

---

*For more information, visit the official website and follow Hugging Face on social media.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
system_prompt_chinese = "You are a chinese interpretor. You are good at translate company brochures from English to Chinese"
user_prompt_chinese = "You will get a stream of English company brochures. Translate them into Chinese"

def stream_brochure_in_chinese(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

    print("\n--- 正在翻译成中文... ---")
    
    translate_stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "你是一位专业的翻译官，请将以下宣传册内容翻译成优雅、地道的中文。"},
            {"role": "user", "content": response}
        ],
        stream=True
    )
    
    chinese_response = ""
    # 我们把中文接在英文后面显示
    for chunk in translate_stream:
        content = chunk.choices[0].delta.content or ""
        chinese_response += content
        full_markdown = f"## English Version\n\n{response}\n\n---\n\n## 中文译文\n\n{chinese_response}"
        update_display(Markdown(full_markdown), display_id=display_handle.display_id)

In [29]:
stream_brochure_in_chinese("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links


## English Version

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the vibrant AI community and collaboration platform shaping the future of machine learning. It serves as the central hub where machine learning engineers, scientists, researchers, and enthusiasts come together to share, explore, and build on open-source models, datasets, and applications.

---

## Our Mission

To empower the next generation of AI practitioners by providing an open, ethical, and collaborative environment where innovation in machine learning can flourish. Hugging Face enables faster development and experimentation through its comprehensive open-source ecosystem.

---

## What We Offer

### Collaboration Platform  
- Host and collaborate on **unlimited public models**, datasets, and applications  
- Access a diverse library of **2 million+ pre-trained models** across text, image, video, audio, and 3D modalities  
- Explore and contribute to over **500,000 datasets** curated by the community  

### Tools & Features  
- **Spaces:** Deploy and demo AI apps and ML models seamlessly  
- **Buckets & Storage:** Efficient data management and sharing  
- **Open Source Stack:** Tools designed to accelerate AI workflows and innovation  
- Build and share your personal ML portfolio to showcase your expertise and projects

---

## Community & Customers

Hugging Face is built by and for a **fast-growing global community** of AI practitioners including researchers, developers, enterprises, and hobbyists. Trusted by leading companies and academia, the platform supports collaboration at every scale — from independent researchers to enterprise AI teams.

---

## Company Culture

At Hugging Face, core values center around:  
- **Open collaboration:** Fostering a welcoming environment to share knowledge freely  
- **Ethical AI:** Commitment to building fair and transparent AI systems  
- **Innovation:** Encouraging experimentation with cutting-edge ML models and techniques  
- **User-first approach:** Designing tools that empower users of all levels to learn and contribute  

The team embraces diversity, transparency, and continuous learning in a remote-friendly work culture.

---

## Careers at Hugging Face

Join us to build the AI community of the future! We seek passionate individuals in areas such as:  
- Machine Learning Engineering  
- Research & Development  
- Software Engineering  
- Developer Relations & Community Management  
- Data Science & Product Management  

Work with a global team dedicated to open-source principles, ethical AI, and cutting-edge research. Be at the forefront of AI innovation while collaborating with a passionate, supportive community.

---

## Join Us Today

Whether you're looking to build AI applications, share your latest models, or connect with the world's leading AI thinkers, Hugging Face offers the tools and community to elevate your machine learning journey.

**Sign up now and start exploring:**  
https://huggingface.co

---

## Brand Identity

- **Colors:**  
  - Hugging Face Yellow: #FFD21E  
  - Accent Orange: #FF9D00  
  - Neutral Gray: #6B7280  

- **Logo & Assets:** Available in .svg, .png, and .ai formats, representing the friendly and open nature of the platform.

---

**Hugging Face — The AI community building the future.**

---

## 中文译文

# Hugging Face 宣传册

---

## 关于 Hugging Face

Hugging Face 是一个充满活力的人工智能社区与合作平台，旨在塑造机器学习的未来。在这里，机器学习工程师、科学家、研究人员和爱好者聚集一堂，分享、探索并构建开源模型、数据集和应用程序。

---

## 我们的使命

我们的使命是通过提供一个开放、道德和协作的环境，赋能下一代人工智能从业者，让机器学习的创新蓬勃发展。Hugging Face 通过其全面的开源生态系统，促进更快速的开发和实验。

---

## 我们提供的服务

### 协作平台  
- 托管和协作开发**无限制的公共模型**、数据集和应用程序  
- 访问超过**200万个预训练模型**，涵盖文本、图像、视频、音频和 3D 模态  
- 探索并为社区策划的**超过 50万个数据集**贡献力量  

### 工具与功能  
- **Spaces:** 无缝部署和演示人工智能应用和机器学习模型  
- **Buckets & Storage:** 高效的数据管理与共享  
- **开源工具栈:** 旨在加速人工智能工作流程和创新的工具  
- 构建并分享个人机器学习作品集，展现您的专业技能和项目

---

## 社区与客户

Hugging Face 是为一个**快速发展的全球人工智能从业者社区**而建立的，包括研究人员、开发者、企业和爱好者。受到领先公司和学术界的信任，该平台支持各个规模的协作——从独立研究人员到企业人工智能团队。

---

## 公司文化

在 Hugging Face，核心价值观体现在：  
- **开放协作：** 促进一个自由分享知识的友好环境  
- **道德人工智能：** 致力于构建公平和透明的人工智能系统  
- **创新：** 鼓励用尖端的机器学习模型和技术进行实验  
- **用户优先：** 设计使所有层次用户都能学习和贡献的工具  

团队在友好的远程工作文化中，接受多样性、透明性和持续学习。

---

## 加入 Hugging Face 的职业机会

加入我们，共同打造未来的人工智能社区！我们寻找对以下领域充满热情的人才：  
- 机器学习工程  
- 研究与开发  
- 软件工程  
- 开发者关系与社区管理  
- 数据科学与产品管理  

与全球团队合作，致力于开源原则、道德人工智能和尖端研究。在与充满激情和支持的社区协作的同时，走在人工智能创新的最前沿。

---

## 今天就加入我们

无论您是希望开发人工智能应用、分享最新模型，还是与全球顶尖的人工智能思想家联系，Hugging Face 都能提供提升您机器学习旅程的工具和社区。

**立即注册，开始探索：**  
https://huggingface.co

---

## 品牌形象

- **颜色：**  
  - Hugging Face 黄色：#FFD21E  
  - 点缀橙：#FF9D00  
  - 中立灰：#6B7280  

- **标志与资产：** 以 .svg、.png 和 .ai 格式提供，彰显平台友好与开放的特性。

---

**Hugging Face — 共同构建未来的人工智能社区。**


--- 正在翻译成中文... ---


In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face  
Hugging Face is the AI community building the future, serving as the premier collaboration platform for the machine learning (ML) community worldwide. Known as "The Home of Machine Learning," Hugging Face empowers engineers, scientists, and AI enthusiasts to create, discover, collaborate on, and share ML models, datasets, and applications.  

With a fast-growing and vibrant community, Hugging Face fosters an open, ethical AI culture by providing a centralized hub to explore and experiment with over **2 million machine learning models** and **500,000+ datasets** across multiple modalities including text, image, audio, video, and even 3D.  

---

## What We Offer  

- **Hugging Face Hub**: Host unlimited public models, datasets, and applications in an open-source ecosystem.  
- **Models**: Browse and contribute to millions of state-of-the-art models, like natural language processing, computer vision, reinforcement learning, and more.  
- **Datasets**: Access and share extensive data collections tailored for diverse ML tasks.  
- **Spaces**: Interactive AI applications running on the platform, enabling easy deployment and sharing of ML demos and projects.  
- **Open Source Stack**: Tools and libraries to accelerate your ML projects and research.  
- **Enterprise Solutions**: Scalable offerings tailored for business application of AI.

---

## Company Culture  
At Hugging Face, the core belief is in **openness, collaboration, and ethical AI development**. The platform nurtures a global community where sharing knowledge and resources accelerates innovation. The company champions a culture that values inclusivity, transparency, and the responsible use of AI technology to positively impact society.

Users are encouraged to build their ML profiles by publishing their work, engaging with others, and contributing to a collective intelligence that drives the field forward.

---

## Our Customers and Community  
Hugging Face serves a broad spectrum of users:  
- **Machine Learning Engineers & Researchers:** Who develop and share models and datasets, pushing the frontier of AI.  
- **Data Scientists & Developers:** Who build applications and solutions using Hugging Face’s tools.  
- **Businesses & Enterprises:** Who leverage the platform’s enterprise features to integrate AI seamlessly at scale.  
- **Educators & Students:** Who use Hugging Face as a learning platform to understand and apply ML.

The platform is trusted by renowned organizations and thousands of global users contributing to and benefiting from a growing ecosystem driven by innovation and shared knowledge.

---

## Careers and Opportunities  
Hugging Face is growing rapidly, offering exciting career opportunities in AI research, engineering, product development, community management, and more. They seek passionate individuals who share a commitment to advancing machine learning and ethical AI. Working at Hugging Face means being part of a mission-driven team that values openness, creativity, impact, and collaboration.  

Interested candidates can explore openings on the company website and join a global team shaping the future of AI.

---

## Get Involved  
- **Explore ML Models and Apps:** Visit [huggingface.co](https://huggingface.co) to browse, experiment, and contribute.  
- **Share Your Work:** Build your public profile by publishing models, datasets, and AI-powered applications.  
- **Join the Community:** Engage with fellow AI enthusiasts to learn, collaborate, and help shape an open AI future.  
- **Sign Up Today:** Start your journey in the home of machine learning and innovation.

---

### Brand Colors  
- Vibrant Yellow: #FFD21E  
- Orange Accent: #FF9D00  
- Neutral Gray: #6B7280  

---

Hugging Face — **Building the Future of AI, Together.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>